In [1]:
import pandas as pd 
import   psycopg2

In [2]:
#Tratamento de dados:                     #Caminho do seu arquivo json
caminho_do_arquivo = r"C:\Users\manue\OneDrive\Desktop\Data-Enginner\Pipeline-de-Dados-ANAC\Anac\Arquivo-Json\PecasAprovadas.json"
df = pd.read_json(caminho_do_arquivo, encoding='utf-8-sig')

In [3]:
#Colunas selecionadas:
colunas = ['ORG_NOME', 'PAPP_COD','PAPP_NOME' ,'PAPP_PN',
       'APAA_CODI', 'APAA_DATA', 'APAA_STATUS']
df = df[colunas]      

In [4]:
df['APAA_DATA'] = pd.to_datetime(df['APAA_DATA'], errors='coerce')
df = df.astype(object).where(df.notnull(), None) #Python None é convertido para NULL pelo driver do banco (psycopg2/npgsql).

In [5]:
#Parametro de conexão:
dbname ='forca_aerea'          
user = 'postgres'
password = 'db123'            
host = 'localhost'
port = '5432'

#Cria uma conexão:
conexao= psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)

#Cria uma cursor  para manipular os dados:
cursor = conexao.cursor()
#Delete base antes da Carga:
#cursor.execute(" delete from Anac")

#Carga dos Dados:
for indice,coluna_df in df.iterrows():
      cursor.execute( """  insert into  ANAC  (
				ORG_NOME,
				PAPP_COD,
				PAPP_NOME,
				PAPP_PN,
			    APAA_CODI,
				APAA_DATA,
				APAA_STATUS
		) VALUES (%s,%s,%s,%s,%s,%s,%s) 

		""", (
				coluna_df["ORG_NOME"],
				coluna_df["PAPP_COD"],
				coluna_df["PAPP_NOME"],
				coluna_df["PAPP_PN"],
				coluna_df["APAA_CODI"],
		    	coluna_df["APAA_DATA"],
				coluna_df["APAA_STATUS"]
		)
		 )
conexao.commit()
cursor.close()
conexao.close()